In [1]:
import pandas as pd
import numpy as np

# Load all 6 clean tables
riders  = pd.read_csv("riders_clean.csv")
mpesa   = pd.read_csv("mpesa_clean.csv",
                      parse_dates=["transaction_date"])
loans   = pd.read_csv("loans_clean.csv")
sacco   = pd.read_csv("sacco_clean.csv")
trips   = pd.read_csv("trips_clean.csv")
apps    = pd.read_csv("apps_clean.csv")

# Confirm everything loaded correctly
print("CLEAN TABLES LOADED")
print()
for name, df in [("riders", riders), ("mpesa", mpesa),
                  ("loans", loans),  ("sacco", sacco),
                  ("trips", trips),  ("apps",  apps)]:
    print(f"  ✅ {name:<8} {len(df):>7,} rows  x  {df.shape[1]:>2} columns")

CLEAN TABLES LOADED

  ✅ riders     1,000 rows  x   8 columns
  ✅ mpesa    203,386 rows  x  17 columns
  ✅ loans      1,390 rows  x   9 columns
  ✅ sacco      5,227 rows  x  11 columns
  ✅ trips     22,828 rows  x   8 columns
  ✅ apps       1,000 rows  x   5 columns


✅ Income features built
   Shape: (1000, 7)
   Riders covered: 1000

Sample output — first 3 riders:


C:\Users\hp\AppData\Local\Temp\ipykernel_8540\1934652305.py:108: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(calc_max_gap)


,rider_id,avg_daily_income,total_income_90d,income_volatility_cv,income_trend_90d,rain_season_dip,income_gap_max_days
0,RDR-00001,912.588409,80307.78,0.179872,0.040904,0.247453,1
1,RDR-00002,950.066744,81705.74,0.198440,0.037983,0.221748,1
2,RDR-00003,1433.113375,114649.07,0.224700,0.039057,0.269743,1



Key statistics:
  Avg daily income:    KES 900
  Avg volatility CV:   0.192
  Avg max gap (days):  1.4


In [3]:
print("Building expense and capacity features...")

MONTHLY_FACTOR = 90 / 30  # convert 90-day totals to monthly equivalent

expense_list = []

for rider_id, rider_txns in mpesa.groupby("rider_id"):

    # Monthly income estimate
    monthly_income = (rider_txns[rider_txns["counterparty_type"] == "INCOME_IN"]
                      ["amount"].sum() / MONTHLY_FACTOR)

    # Fuel spend
    fuel_monthly = (rider_txns[rider_txns["counterparty_type"] == "FUEL"]
                    ["amount"].sum() / MONTHLY_FACTOR)

    # Fuel to income ratio
    # How much of income goes to fuel?
    # Above 0.40 is a warning sign
    fuel_ratio = fuel_monthly / monthly_income if monthly_income > 0 else 0

    # SACCO contributions
    sacco_monthly = (rider_txns[rider_txns["counterparty_type"] == "SACCO_CONTRIB"]
                     ["amount"].sum() / MONTHLY_FACTOR)

    # Family remittances
    remit_monthly = (rider_txns[rider_txns["counterparty_type"] == "FAMILY_SEND"]
                     ["amount"].sum() / MONTHLY_FACTOR)

    # Digital loan outflows
    digital_monthly = (rider_txns[rider_txns["counterparty_type"] == "DIGITAL_LOAN"]
                       ["amount"].sum() / MONTHLY_FACTOR)

    # Net Disposable Income
    # Income minus all known expenses
    # We add KES 3,000 as food estimate for every rider
    total_expenses = fuel_monthly + sacco_monthly + remit_monthly + 3000
    estimated_ndi  = max(monthly_income - total_expenses, 1000)

    # Debt service ratio
    # What fraction of income is going to existing loan repayments?
    # Above 0.50 triggers the over-indebtedness hard block
    debt_obligations   = sacco_monthly + digital_monthly
    debt_service_ratio = min(
        debt_obligations / monthly_income if monthly_income > 0 else 0,
        1.0
    )

    # Max safe repayment
    # 30% of NDI — this is the ceiling for any loan repayment we recommend
    max_safe_repayment = estimated_ndi * 0.30

    # M-Pesa balance health
    avg_balance = rider_txns["running_balance"].mean()
    min_balance = rider_txns["running_balance"].min()

    # Zero balance days
    # How many days did the balance drop below KES 50?
    rider_copy = rider_txns.copy()
    rider_copy["date"] = rider_copy["transaction_date"].dt.date
    daily_last_balance = (rider_copy
                          .sort_values("transaction_date")
                          .groupby("date")["running_balance"]
                          .last())
    zero_balance_days = int((daily_last_balance < 50).sum())

    expense_list.append({
        "rider_id":                  rider_id,
        "monthly_income_estimate":   round(monthly_income, 2),
        "fuel_spend_monthly":        round(fuel_monthly, 2),
        "fuel_to_income_ratio":      round(fuel_ratio, 4),
        "sacco_contrib_monthly":     round(sacco_monthly, 2),
        "family_remittance_monthly": round(remit_monthly, 2),
        "digital_loan_monthly_out":  round(digital_monthly, 2),
        "estimated_ndi":             round(estimated_ndi, 2),
        "debt_service_ratio":        round(debt_service_ratio, 4),
        "max_safe_repayment":        round(max_safe_repayment, 2),
        "avg_mpesa_balance":         round(avg_balance, 2),
        "min_mpesa_balance":         round(min_balance, 2),
        "zero_balance_days":         zero_balance_days,
    })

expense_features = pd.DataFrame(expense_list)

print(f"✅ Expense features built")
print(f"   Shape: {expense_features.shape}")
print()
print("Sample output — first 3 riders:")
display(expense_features.head(3))
print()
print("Key statistics:")
print(f"  Avg monthly income:     KES {expense_features.monthly_income_estimate.mean():.0f}")
print(f"  Avg estimated NDI:      KES {expense_features.estimated_ndi.mean():.0f}")
print(f"  Avg max safe repayment: KES {expense_features.max_safe_repayment.mean():.0f}")
print(f"  Avg debt service ratio: {expense_features.debt_service_ratio.mean():.1%}")
print(f"  Avg zero balance days:  {expense_features.zero_balance_days.mean():.1f}")

Building expense and capacity features...
✅ Expense features built
   Shape: (1000, 13)

Sample output — first 3 riders:


,rider_id,monthly_income_estimate,fuel_spend_monthly,fuel_to_income_ratio,sacco_contrib_monthly,family_remittance_monthly,digital_loan_monthly_out,estimated_ndi,debt_service_ratio,max_safe_repayment,avg_mpesa_balance,min_mpesa_balance,zero_balance_days
0,RDR-00001,26769.26,8728.20,0.3261,3091.70,0.0,1194.76,11949.36,0.1601,3584.81,20193.22,2337.65,0
1,RDR-00002,27235.25,7151.31,0.2626,4723.98,0.0,522.29,12359.95,0.1926,3707.99,21003.40,1921.34,0
2,RDR-00003,38216.36,7600.43,0.1989,4074.63,0.0,839.83,23541.29,0.1286,7062.39,34745.82,1411.60,0



Key statistics:
  Avg monthly income:     KES 24822
  Avg estimated NDI:      KES 10509
  Avg max safe repayment: KES 3153
  Avg debt service ratio: 16.0%
  Avg zero balance days:  0.0


In [4]:
expense_features.head()

,rider_id,monthly_income_estimate,fuel_spend_monthly,fuel_to_income_ratio,sacco_contrib_monthly,family_remittance_monthly,digital_loan_monthly_out,estimated_ndi,debt_service_ratio,max_safe_repayment,avg_mpesa_balance,min_mpesa_balance,zero_balance_days
0,RDR-00001,26769.26,8728.20,0.3261,3091.70,0.00,1194.76,11949.36,0.1601,3584.81,20193.22,2337.65,0
1,RDR-00002,27235.25,7151.31,0.2626,4723.98,0.00,522.29,12359.95,0.1926,3707.99,21003.40,1921.34,0
2,RDR-00003,38216.36,7600.43,0.1989,4074.63,0.00,839.83,23541.29,0.1286,7062.39,34745.82,1411.60,0
3,RDR-00004,35633.80,6679.76,0.1875,1874.60,0.00,406.10,24079.44,0.0640,7223.83,36187.75,1989.37,0
4,RDR-00005,23638.38,6983.09,0.2954,2174.47,2934.37,0.00,8546.45,0.0920,2563.94,15034.41,1592.96,0


In [5]:
expense_features["min_mpesa_balance"].agg(["min", "max"])

min       0.00
max    3131.33
Name: min_mpesa_balance, dtype: float64

In [29]:
sacco_list = []

for rider_id, rider_sacco in sacco.groupby("rider_id"):

    tenure = rider_sacco["months_as_member"].max()

    contribution_rate = rider_sacco["contribution_made"].mean()

    made = rider_sacco[rider_sacco["contribution_made"] == 1]
    on_time_rate = made["contribution_on_time"].mean() if len(made) > 0 else 0

    # FIXED lateness
    late = rider_sacco[rider_sacco["days_late"] > 0]
    avg_days_late = late["days_late"].mean() if len(late) > 0 else 0
    late_rate = (rider_sacco["days_late"] > 0).mean()

    cumulative_savings = rider_sacco["cumulative_savings"].max()
    guarantor_count = rider_sacco["guarantor_count"].max()
   

    sacco_list.append({
        "rider_id": rider_id,
        "sacco_tenure_months": int(tenure),
        "sacco_contribution_rate": round(float(contribution_rate), 4),
        "sacco_on_time_rate": round(float(on_time_rate), 4),
        "sacco_late_rate": round(float(late_rate), 4),   # NEW
        "sacco_avg_days_late": round(float(avg_days_late), 2),
        "sacco_cumulative_savings": round(float(cumulative_savings), 2),
        "sacco_guarantor_count": int(guarantor_count),
        "sacco_loan_stress_ratio": round(float(loan_stress_ratio), 4),
    })

    sacco_features = pd.DataFrame(sacco_list)

In [30]:
print(sacco_features.columns.tolist())

['rider_id', 'sacco_tenure_months', 'sacco_contribution_rate', 'sacco_on_time_rate', 'sacco_late_rate', 'sacco_avg_days_late', 'sacco_cumulative_savings', 'sacco_guarantor_count', 'sacco_loan_stress_ratio']


In [8]:
print(sacco.columns.tolist())

['rider_id', 'months_as_member', 'monthly_contribution', 'contribution_made', 'contribution_on_time', 'days_late', 'cumulative_savings', 'shares_owned', 'active_loan_balance', 'guarantor_count', 'repayment_status_code']


In [31]:
loan_stress_ratio = rider_sacco["active_loan_balance"].sum() / (
    rider_sacco["monthly_contribution"].sum() + 1e-6
)

In [32]:


loan_list = []

for rider_id in riders["rider_id"].unique():

    rider_loans = loans[loans["rider_id"] == rider_id]

   
    if len(rider_loans) == 0:
        loan_list.append({
            "rider_id":               rider_id,
            "total_loans_taken":      0,
            "loans_repaid_clean":     0,
            "on_time_repayment_rate": 0.75,  # neutral — not punished
            "avg_days_early_late":    0.0,
            "max_loan_repaid":        0.0,
            "active_digital_loans":   0,
            "digital_loan_frequency": 0,
            "ever_defaulted":         0,
            "restructured_loan_flag": 0,
            "total_outstanding_debt": 0.0,
            "first_time_borrower":    1,
        })
        continue

    # Total loans ever taken
    total_loans = len(rider_loans)

    # Clean repayments — completed and paid within 7 days of due date
    completed   = rider_loans[
        (rider_loans["is_completed"] == 1) &
        (rider_loans["default_flag"] == 0)
    ]
    clean_count  = int((completed["days_early_late"] <= 7).sum())
    on_time_rate = clean_count / total_loans if total_loans > 0 else 0.75

    # Average days early or late
    # Negative = tends to pay early (good signal)
    # Positive = tends to pay late (risk signal)
    avg_days = rider_loans["days_early_late"].mean()

    # Largest loan ever successfully repaid
    # This tells us the proven repayment ceiling
    clean_loans  = rider_loans[
        (rider_loans["default_flag"] == 0) &
        (rider_loans["is_completed"] == 1)
    ]
    max_repaid   = clean_loans["loan_amount"].max() if len(clean_loans) > 0 else 0.0

    # Digital loan activity
    digital      = rider_loans[rider_loans["lender_type_code"] == 3]
    active_digital = int((digital["outstanding_balance"] > 0).sum())
    digital_freq   = len(digital)

    # Ever defaulted on any loan
    ever_defaulted   = int(rider_loans["default_flag"].any())
    restructured     = int(rider_loans["restructured_flag"].any())

    # Total debt still outstanding across all lenders
    total_outstanding = rider_loans["outstanding_balance"].sum()

    loan_list.append({
        "rider_id":               rider_id,
        "total_loans_taken":      total_loans,
        "loans_repaid_clean":     clean_count,
        "on_time_repayment_rate": round(on_time_rate, 4),
        "avg_days_early_late":    round(float(avg_days), 2),
        "max_loan_repaid":        round(float(max_repaid), 2),
        "active_digital_loans":   active_digital,
        "digital_loan_frequency": digital_freq,
        "ever_defaulted":         ever_defaulted,
        "restructured_loan_flag": restructured,
        "total_outstanding_debt": round(float(total_outstanding), 2),
        "first_time_borrower":    0,
    })

loan_features = pd.DataFrame(loan_list)

print(f"✅ Loan history features built")
print(f"   Shape: {loan_features.shape}")
print()
print("Key statistics:")
print(f"  First time borrowers:      {loan_features.first_time_borrower.sum()}")
print(f"  Ever defaulted:            {loan_features.ever_defaulted.sum()}")
print(f"  Avg on-time rate:          {loan_features.on_time_repayment_rate.mean():.1%}")
print(f"  Riders with digital loans: {(loan_features.active_digital_loans > 0).sum()}")
print()
display(loan_features.head(3))

✅ Loan history features built
   Shape: (1000, 12)

Key statistics:
  First time borrowers:      294
  Ever defaulted:            46
  Avg on-time rate:          66.8%
  Riders with digital loans: 26



,rider_id,total_loans_taken,loans_repaid_clean,on_time_repayment_rate,avg_days_early_late,max_loan_repaid,active_digital_loans,digital_loan_frequency,ever_defaulted,restructured_loan_flag,total_outstanding_debt,first_time_borrower
0,RDR-00001,0,0,0.75,0.00,0.00,0,0,0,0,0.00,1
1,RDR-00002,0,0,0.75,0.00,0.00,0,0,0,0,0.00,1
2,RDR-00003,4,1,0.25,9.75,27601.41,0,2,0,0,24751.76,0


In [21]:
loans.loc[loans["total_loans_taken"] == 0, "on_time_repayment_rate"] = 0
loans.loc[loans["total_loans_taken"] == 0, "avg_days_early_late"] = 0
loans.loc[loans["total_loans_taken"] == 0, "loans_repaid_clean"] = 0

KeyError: 'total_loans_taken'

In [33]:


# ── Part A: From riders table ────────────────────────────────────────────────
behaviour_features = riders[[
    "rider_id", "age", "months_operating",
    "bike_age_years", "bike_owned",
    "has_app", "segment_risk_score"
]].copy()

# ── Part B: From app trips table ─────────────────────────────────────────────
app_list = []

for rider_id, rider_trips in trips.groupby("rider_id"):

    weekly_trips    = rider_trips["total_trips"].mean()
    weekly_earnings = rider_trips["gross_earnings_week"].mean()
    avg_rating      = rider_trips["platform_rating"].mean()
    avg_cancel      = rider_trips["cancellation_rate"].mean()
    avg_active_days = rider_trips["active_days"].mean()
    avg_peak_ratio  = rider_trips["peak_hour_ratio"].mean()

    # Income stability from app earnings
    # Lower CV = more stable weekly earnings = better signal
    if rider_trips["gross_earnings_week"].mean() > 0:
        earnings_cv = (rider_trips["gross_earnings_week"].std() /
                       rider_trips["gross_earnings_week"].mean())
    else:
        earnings_cv = 1.0

    app_list.append({
        "rider_id":                rider_id,
        "app_avg_weekly_trips":    round(weekly_trips, 1),
        "app_avg_weekly_earnings": round(weekly_earnings, 2),
        "app_platform_rating":     round(avg_rating, 2),
        "app_cancellation_rate":   round(avg_cancel, 4),
        "app_active_days_avg":     round(avg_active_days, 2),
        "app_peak_hour_ratio":     round(float(avg_peak_ratio), 4),
        "app_income_stability_cv": round(float(earnings_cv), 4),
    })

app_features_df = pd.DataFrame(app_list)

# Calculate medians BEFORE merging
# Non-app riders get these medians — not zeros
# This is the fairness rule from Step 6
app_medians = {
    col: app_features_df[col].median()
    for col in app_features_df.columns
    if col != "rider_id"
}

print("Medians used for non-app riders:")
for col, val in app_medians.items():
    print(f"  {col:<30} {val:.3f}")

# Merge app features onto behaviour features
behaviour_features = behaviour_features.merge(
    app_features_df, on="rider_id", how="left"
)

# Fill non-app riders with population medians
for col, median_val in app_medians.items():
    behaviour_features[col] = behaviour_features[col].fillna(median_val)

print(f"\n Behavioural features built")
print(f"   Shape: {behaviour_features.shape}")
print(f"   App riders:     {behaviour_features.has_app.sum()}")
print(f"   Non-app riders: {(behaviour_features.has_app == 0).sum()}")
print()
display(behaviour_features.head(3))

Medians used for non-app riders:
  app_avg_weekly_trips           35.100
  app_avg_weekly_earnings        5285.140
  app_platform_rating            4.450
  app_cancellation_rate          0.065
  app_active_days_avg            5.500
  app_peak_hour_ratio            0.434
  app_income_stability_cv        0.302

 Behavioural features built
   Shape: (1000, 14)
   App riders:     439
   Non-app riders: 561



,rider_id,age,months_operating,bike_age_years,bike_owned,has_app,segment_risk_score,app_avg_weekly_trips,app_avg_weekly_earnings,app_platform_rating,app_cancellation_rate,app_active_days_avg,app_peak_hour_ratio,app_income_stability_cv
0,RDR-00001,30,62,5.0,1,1,4,38.9,5833.81,4.72,0.0664,5.58,0.4411,0.3205
1,RDR-00002,27,15,1.7,1,0,5,35.1,5285.14,4.45,0.0648,5.50,0.4344,0.3019
2,RDR-00003,35,4,7.8,1,1,3,38.3,5884.92,4.55,0.0649,5.56,0.4231,0.2549


In [34]:
print("Columns in trips DataFrame:")
print(trips.columns.tolist())

Columns in trips DataFrame:
['rider_id', 'total_trips', 'active_days', 'gross_earnings_week', 'platform_rating', 'cancellation_rate', 'peak_hour_trips', 'longest_inactive_streak', 'peak_hour_ratio']


In [35]:


trips["peak_hour_ratio"] = (
    trips["peak_hour_trips"] /
    trips["total_trips"].replace(0, 1)  # avoid division by zero
).round(4)

print("✅ peak_hour_ratio added")
print()
print("Columns now in trips:")
print(trips.columns.tolist())
print()
print("Sample values:")
print(trips["peak_hour_ratio"].describe())

✅ peak_hour_ratio added

Columns now in trips:
['rider_id', 'total_trips', 'active_days', 'gross_earnings_week', 'platform_rating', 'cancellation_rate', 'peak_hour_trips', 'longest_inactive_streak', 'peak_hour_ratio']

Sample values:
count    22828.000000
mean         0.434015
std          0.059012
min          0.285700
25%          0.384600
50%          0.434800
75%          0.483900
max          0.548400
Name: peak_hour_ratio, dtype: float64


In [36]:


# ── Step 1: Merge all 5 feature families ────────────────────────────────────
feature_matrix = (income_features
                  .merge(expense_features,   on="rider_id", how="inner")
                  .merge(sacco_features,     on="rider_id", how="inner")
                  .merge(loan_features,      on="rider_id", how="inner")
                  .merge(behaviour_features, on="rider_id", how="inner"))

print(f"After merging all feature families: {feature_matrix.shape}")

# ── Step 2: Attach the target variable ──────────────────────────────────────
# actual_default is what we are predicting
# requested_amount and requested_term_days give useful context
target = apps[["rider_id", "actual_default",
               "requested_amount", "requested_term_days",
               "loan_purpose_code"]]

feature_matrix = feature_matrix.merge(target, on="rider_id", how="inner")

print(f"After attaching target variable: {feature_matrix.shape}")

# ── Step 3: Check for missing values ────────────────────────────────────────
total_missing = feature_matrix.isnull().sum().sum()
print(f"\nMissing values: {total_missing}")

if total_missing > 0:
    print("Filling remaining NaN with 0...")
    feature_matrix = feature_matrix.fillna(0)

# ── Step 4: Clip extreme outliers ────────────────────────────────────────────
# Cap at 1st and 99th percentile
# Prevents one extreme rider from distorting the model
skip_cols    = ["rider_id", "actual_default",
                "requested_amount", "requested_term_days"]
numeric_cols = [c for c in feature_matrix.select_dtypes(include=[np.number]).columns
                if c not in skip_cols]

for col in numeric_cols:
    p01 = feature_matrix[col].quantile(0.01)
    p99 = feature_matrix[col].quantile(0.99)
    feature_matrix[col] = feature_matrix[col].clip(p01, p99)

print(f"\n✅ Feature matrix assembled")
print(f"   Shape: {feature_matrix.shape[0]} riders × {feature_matrix.shape[1]} columns")
print(f"   Default rate: {feature_matrix.actual_default.mean():.1%}")
print(f"   Defaulters: {feature_matrix.actual_default.sum()}")
print(f"   Non-defaulters: {(feature_matrix.actual_default == 0).sum()}")
print()
display(feature_matrix.head(3))

After merging all feature families: (369, 51)
After attaching target variable: (369, 55)

Missing values: 0

✅ Feature matrix assembled
   Shape: 369 riders × 55 columns
   Default rate: 3.0%
   Defaulters: 11
   Non-defaulters: 358



,rider_id,avg_daily_income,total_income_90d,income_volatility_cv,income_trend_90d,rain_season_dip,income_gap_max_days,monthly_income_estimate,fuel_spend_monthly,fuel_to_income_ratio,...,app_avg_weekly_earnings,app_platform_rating,app_cancellation_rate,app_active_days_avg,app_peak_hour_ratio,app_income_stability_cv,actual_default,requested_amount,requested_term_days,loan_purpose_code
0,RDR-00001,912.588409,80307.78,0.179872,0.040904,0.247453,1.0,26769.26,8728.20,0.3261,...,5833.81,4.72,0.0664,5.58,0.4411,0.3205,0,20000,180,3
1,RDR-00002,950.066744,81705.74,0.198440,0.037983,0.221748,1.0,27235.25,7151.31,0.2626,...,5285.14,4.45,0.0648,5.50,0.4344,0.3019,0,5000,30,5
2,RDR-00005,875.495432,70915.13,0.184480,0.044000,0.227222,2.0,23638.38,6983.09,0.2954,...,5285.14,4.45,0.0648,5.50,0.4344,0.3019,0,5000,180,1


In [37]:
import numpy as np

# =========================
# 1. Loan-to-income ratio
# =========================
feature_matrix["loan_to_income_ratio"] = (
    feature_matrix["requested_amount"] /
    np.where(feature_matrix["monthly_income_estimate"] == 0, 1, feature_matrix["monthly_income_estimate"])
).round(4)

# =========================
# 2. Loan-to-NDI ratio
# =========================
feature_matrix["loan_to_ndi_ratio"] = (
    feature_matrix["requested_amount"] /
    np.where(feature_matrix["estimated_ndi"] == 0, 1, feature_matrix["estimated_ndi"])
).round(4)

# =========================
# 3. Monthly repayment estimate
# =========================
feature_matrix["requested_monthly_repayment"] = (
    feature_matrix["requested_amount"] /
    np.where(feature_matrix["requested_term_days"] == 0, 1, feature_matrix["requested_term_days"] / 30)
).round(2)

# =========================
# 4. Repayment pressure ratio
# =========================
feature_matrix["repayment_to_ndi_ratio"] = (
    feature_matrix["requested_monthly_repayment"] /
    np.where(feature_matrix["estimated_ndi"] == 0, 1, feature_matrix["estimated_ndi"])
).round(4)

# =========================
# 5. Overborrowing flag
# =========================
feature_matrix["overborrowing_flag"] = (
    feature_matrix["repayment_to_ndi_ratio"] > 0.30
).astype(int)

# =========================
# Summary stats
# =========================
print("✅ Loan sizing risk features added\n")

print(f"Avg loan-to-income ratio: {feature_matrix['loan_to_income_ratio'].mean():.2f}x")
print(f"Avg loan-to-NDI ratio:    {feature_matrix['loan_to_ndi_ratio'].mean():.2f}x")
print(f"Overborrowing detected:   {feature_matrix['overborrowing_flag'].sum()} riders")
print(f"Overborrowing rate:       {feature_matrix['overborrowing_flag'].mean():.1%}")

print("\nDefault rate by overborrowing flag:")
print(
    feature_matrix.groupby("overborrowing_flag")["actual_default"]
    .mean()
    .round(3)
)

✅ Loan sizing risk features added

Avg loan-to-income ratio: 0.83x
Avg loan-to-NDI ratio:    3.20x
Overborrowing detected:   289 riders
Overborrowing rate:       78.3%

Default rate by overborrowing flag:
overborrowing_flag
0    0.038
1    0.028
Name: actual_default, dtype: float64


In [40]:
feature_matrix.head()


,rider_id,avg_daily_income,total_income_90d,income_volatility_cv,income_trend_90d,rain_season_dip,income_gap_max_days,monthly_income_estimate,fuel_spend_monthly,fuel_to_income_ratio,...,app_income_stability_cv,actual_default,requested_amount,requested_term_days,loan_purpose_code,loan_to_income_ratio,loan_to_ndi_ratio,requested_monthly_repayment,repayment_to_ndi_ratio,overborrowing_flag
0,RDR-00001,912.588409,80307.78,0.179872,0.040904,0.247453,1.0,26769.26,8728.20,0.3261,...,0.3205,0,20000,180,3,0.7471,1.6737,3333.33,0.2790,0
1,RDR-00002,950.066744,81705.74,0.198440,0.037983,0.221748,1.0,27235.25,7151.31,0.2626,...,0.3019,0,5000,30,5,0.1836,0.4045,5000.00,0.4045,1
2,RDR-00005,875.495432,70915.13,0.184480,0.044000,0.227222,2.0,23638.38,6983.09,0.2954,...,0.3019,0,5000,180,1,0.2115,0.5850,833.33,0.0975,0
3,RDR-00007,764.972326,65787.62,0.207398,0.022626,0.240342,1.0,21929.21,6470.36,0.2951,...,0.3019,0,10000,180,2,0.4560,1.0544,1666.67,0.1757,0
4,RDR-00008,837.035181,69473.92,0.232879,0.016778,0.255329,3.0,23157.97,7736.89,0.3341,...,0.2918,0,10000,180,1,0.4318,2.5433,1666.67,0.4239,1


In [43]:
object_cols = [
    c for c in feature_matrix.columns
    if feature_matrix[c].dtype == "object" and c != "rider_id"
]
print(f"  Text columns remaining: {len(object_cols)}")
if len(object_cols) > 0:
    print(f"  ⚠️  Fix these: {object_cols}")
else:
    print(f"  ✅ All columns are numeric")
print()

# Show feature families
income_cols   = [c for c in feature_matrix if any(x in c for x in
                 ["income","inflow","gap","uplift","trend","rain"])]
expense_cols  = [c for c in feature_matrix if any(x in c for x in
                 ["fuel","ndi","debt","balance","remit","digital_loan",
                  "repayment","safe"])]
sacco_cols    = [c for c in feature_matrix if "sacco" in c or "savings" in c]
loan_cols     = [c for c in feature_matrix if any(x in c for x in
                 ["loan","default","restructured","outstanding","borrower"])]
behav_cols    = [c for c in feature_matrix if any(x in c for x in
                 ["age","months_op","bike","app_","segment","has_app",
                  "peak","cancel","rating"])]

print("  FEATURES BY FAMILY:")
print(f"    Income features:       {len(income_cols)}")
print(f"    Expense features:      {len(expense_cols)}")
print(f"    Savings features:      {len(sacco_cols)}")
print(f"    Loan history features: {len(loan_cols)}")
print(f"    Behavioural features:  {len(behav_cols)}")
print()

# Show overborrowing insight you added
print("  OVERBORROWING CHECK:")
print(f"    Riders overborrowing:  {feature_matrix.overborrowing_flag.sum()}")
print(f"    Default rate — overborrowing:     "
      f"{feature_matrix[feature_matrix.overborrowing_flag==1].actual_default.mean():.1%}")
print(f"    Default rate — not overborrowing: "
      f"{feature_matrix[feature_matrix.overborrowing_flag==0].actual_default.mean():.1%}")
print()
print("  ✅ Feature matrix is ready for model training")
print("=" * 55)

  Text columns remaining: 0
  ✅ All columns are numeric

  FEATURES BY FAMILY:
    Income features:       10
    Expense features:      17
    Savings features:      9
    Loan history features: 15
    Behavioural features:  13

  OVERBORROWING CHECK:
    Riders overborrowing:  289
    Default rate — overborrowing:     2.8%
    Default rate — not overborrowing: 3.8%

  ✅ Feature matrix is ready for model training


In [44]:
print("FINAL FEATURE MATRIX CHECK")
print()
print(f"Shape: {feature_matrix.shape}")
print(f"Default rate: {feature_matrix.actual_default.mean():.1%}")
print(f"Missing values: {feature_matrix.isnull().sum().sum()}")
print()
print("All columns:")
for col in feature_matrix.columns:
    print(f"  {col}")

FINAL FEATURE MATRIX CHECK

Shape: (369, 60)
Default rate: 3.0%
Missing values: 0

All columns:
  rider_id
  avg_daily_income
  total_income_90d
  income_volatility_cv
  income_trend_90d
  rain_season_dip
  income_gap_max_days
  monthly_income_estimate
  fuel_spend_monthly
  fuel_to_income_ratio
  sacco_contrib_monthly
  family_remittance_monthly
  digital_loan_monthly_out
  estimated_ndi
  debt_service_ratio
  max_safe_repayment
  avg_mpesa_balance
  min_mpesa_balance
  zero_balance_days
  sacco_tenure_months
  sacco_contribution_rate
  sacco_on_time_rate
  sacco_late_rate
  sacco_avg_days_late
  sacco_cumulative_savings
  sacco_guarantor_count
  sacco_loan_stress_ratio
  total_loans_taken
  loans_repaid_clean
  on_time_repayment_rate
  avg_days_early_late
  max_loan_repaid
  active_digital_loans
  digital_loan_frequency
  ever_defaulted
  restructured_loan_flag
  total_outstanding_debt
  first_time_borrower
  age
  months_operating
  bike_age_years
  bike_owned
  has_app
  segment_ri

In [51]:
# Save
feature_matrix.to_csv("features.csv", index=False)

import json
with open("feature_cols.json", "w") as f:
    json.dump(feature_matrix, f, indent=2)



TypeError: Object of type DataFrame is not JSON serializable

In [ ]:
FEATURE_COLS = feature_matrix.columns.tolist()

In [ ]:
feature_matrix["loan_to_income_90d_ratio"] = (
    feature_matrix["requested_amount"] / feature_matrix["total_income_90d"]
)

In [ ]:
feature_matrix.head()

In [ ]:
feature_matrix = feature_matrix.drop(columns=["rider_id"])

In [ ]:
# Add overborrowing features
feature_matrix["loan_to_income_ratio"] = (
    feature_matrix["requested_amount"] /
    feature_matrix["monthly_income_estimate"].replace(0, 1)
).round(4)

feature_matrix["loan_to_ndi_ratio"] = (
    feature_matrix["requested_amount"] /
    feature_matrix["estimated_ndi"].replace(0, 1)
).round(4)

feature_matrix["requested_monthly_repayment"] = (
    feature_matrix["requested_amount"] /
    (feature_matrix["requested_term_days"] / 30).replace(0, 1)
).round(2)

feature_matrix["repayment_to_ndi_ratio"] = (
    feature_matrix["requested_monthly_repayment"] /
    feature_matrix["estimated_ndi"].replace(0, 1)
).round(4)

feature_matrix["overborrowing_flag"] = (
    feature_matrix["repayment_to_ndi_ratio"] > 0.30
).astype(int)

print("✅ Overborrowing features added")
print()
print(f"  Overborrowing riders:  {feature_matrix.overborrowing_flag.sum()}")
print(f"  Default rate — overborrowing:     "
      f"{feature_matrix[feature_matrix.overborrowing_flag==1].actual_default.mean():.1%}")
print(f"  Default rate — not overborrowing: "
      f"{feature_matrix[feature_matrix.overborrowing_flag==0].actual_default.mean():.1%}")

In [ ]:
from sklearn.tree import DecisionTreeClassifier

stump = DecisionTreeClassifier(max_depth=1, random_state=42)
stump.fit(
    feature_matrix[["repayment_to_ndi_ratio"]],
    feature_matrix["actual_default"]
)

threshold = stump.tree_.threshold[0]
print(f"Data-driven threshold: {threshold:.4f}")

feature_matrix["overborrowing_flag"] = (
    feature_matrix["repayment_to_ndi_ratio"] > threshold
).astype(int)

print(f"Overborrowing rate: {feature_matrix['overborrowing_flag'].mean():.1%}")
print("\nDefault rate by overborrowing flag:")
print(feature_matrix.groupby("overborrowing_flag")["actual_default"].mean().round(3))

In [ ]:
print("REPAYMENT TO NDI RATIO — DISTRIBUTION")
print()
print(feature_matrix["repayment_to_ndi_ratio"].describe())
print()

# What fraction of riders are above different thresholds?
for threshold in [0.20, 0.25, 0.30, 0.35, 0.40, 0.50]:
    count = (feature_matrix["repayment_to_ndi_ratio"] > threshold).sum()
    pct   = count / len(feature_matrix) * 100
    print(f"  Above {threshold:.0%}:  {count} riders  ({pct:.0f}%)")

In [ ]:
# Diagnose the problem
print("DIAGNOSIS")
print()

# Check a sample of riders
sample = feature_matrix[[
    "requested_amount",
    "requested_term_days",
    "requested_monthly_repayment",
    "estimated_ndi",
    "repayment_to_ndi_ratio"
]].head(10)

print("Sample riders:")
display(sample)
print()

# Check the requested amounts
print("requested_amount distribution:")
print(feature_matrix["requested_amount"].describe())
print()

# Check NDI
print("estimated_ndi distribution:")
print(feature_matrix["estimated_ndi"].describe())
print()

# Check term days
print("requested_term_days distribution:")
print(feature_matrix["requested_term_days"].describe())

In [52]:
feature_matrix.head()

,rider_id,avg_daily_income,total_income_90d,income_volatility_cv,income_trend_90d,rain_season_dip,income_gap_max_days,monthly_income_estimate,fuel_spend_monthly,fuel_to_income_ratio,...,app_income_stability_cv,actual_default,requested_amount,requested_term_days,loan_purpose_code,loan_to_income_ratio,loan_to_ndi_ratio,requested_monthly_repayment,repayment_to_ndi_ratio,overborrowing_flag
0,RDR-00001,912.588409,80307.78,0.179872,0.040904,0.247453,1.0,26769.26,8728.20,0.3261,...,0.3205,0,20000,180,3,0.7471,1.6737,3333.33,0.2790,0
1,RDR-00002,950.066744,81705.74,0.198440,0.037983,0.221748,1.0,27235.25,7151.31,0.2626,...,0.3019,0,5000,30,5,0.1836,0.4045,5000.00,0.4045,1
2,RDR-00005,875.495432,70915.13,0.184480,0.044000,0.227222,2.0,23638.38,6983.09,0.2954,...,0.3019,0,5000,180,1,0.2115,0.5850,833.33,0.0975,0
3,RDR-00007,764.972326,65787.62,0.207398,0.022626,0.240342,1.0,21929.21,6470.36,0.2951,...,0.3019,0,10000,180,2,0.4560,1.0544,1666.67,0.1757,0
4,RDR-00008,837.035181,69473.92,0.232879,0.016778,0.255329,3.0,23157.97,7736.89,0.3341,...,0.2918,0,10000,180,1,0.4318,2.5433,1666.67,0.4239,1


In [53]:
# Find extreme outliers
print("RIDERS WITH EXTREME RATIOS")
print()

extreme = feature_matrix[
    feature_matrix["repayment_to_ndi_ratio"] > 5
][[
    "requested_amount",
    "requested_term_days",
    "requested_monthly_repayment",
    "estimated_ndi",
    "repayment_to_ndi_ratio"
]]

print(f"Riders with ratio above 5: {len(extreme)}")
display(extreme)
print()

# What is causing extreme NDI?
# Find riders with very low NDI
low_ndi = feature_matrix[feature_matrix["estimated_ndi"] < 2000][[
    "monthly_income_estimate",
    "fuel_spend_monthly",
    "estimated_ndi"
]]
print(f"Riders with NDI below KES 2,000: {len(low_ndi)}")
display(low_ndi.head(10))

RIDERS WITH EXTREME RATIOS

Riders with ratio above 5: 15


,requested_amount,requested_term_days,requested_monthly_repayment,estimated_ndi,repayment_to_ndi_ratio
20,25000,90,8333.33,1000.00,8.3333
30,50000,30,50000.00,5860.02,8.5324
39,25000,30,25000.00,4318.91,5.7885
56,25000,30,25000.00,2044.60,12.2273
113,30000,30,30000.00,1000.00,30.0000
154,50000,30,50000.00,1000.00,50.0000
157,20000,30,20000.00,2956.96,6.7637
166,50000,60,25000.00,3550.30,7.0417
187,30000,30,30000.00,2493.42,12.0317
256,20000,90,6666.67,1000.00,6.6667



Riders with NDI below KES 2,000: 16


,monthly_income_estimate,fuel_spend_monthly,estimated_ndi
17,17301.7900,6098.59,1826.05
20,16237.3684,8521.34,1000.00
113,17002.8200,8620.39,1000.00
125,17339.5600,6877.61,1296.41
141,17004.0600,8645.43,1850.58
154,19360.1600,7675.35,1000.00
175,16551.7700,7997.66,1756.67
190,17266.6200,6071.16,1834.63
225,17480.3000,6962.56,1675.31
231,19259.7900,6571.17,1429.29


In [54]:
# Cap repayment_to_ndi_ratio at 5.0
# Anything above 5 means the same thing — completely unaffordable
# The exact value above 5 does not add information

feature_matrix["repayment_to_ndi_ratio"] = (
    feature_matrix["repayment_to_ndi_ratio"].clip(upper=5.0)
)

# Same for loan_to_ndi_ratio
feature_matrix["loan_to_ndi_ratio"] = (
    feature_matrix["loan_to_ndi_ratio"].clip(upper=10.0)
)

# Same for loan_to_income_ratio
feature_matrix["loan_to_income_ratio"] = (
    feature_matrix["loan_to_income_ratio"].clip(upper=10.0)
)

print("✅ Ratios capped at sensible maximums")
print()
print("repayment_to_ndi_ratio after capping:")
print(feature_matrix["repayment_to_ndi_ratio"].describe())
print()

# Now check distribution again
print("Distribution after capping:")
for threshold in [0.30, 0.50, 1.00, 2.00]:
    count = (feature_matrix["repayment_to_ndi_ratio"] > threshold).sum()
    pct   = count / len(feature_matrix) * 100
    print(f"  Above {threshold:.2f}:  {count} riders  ({pct:.0f}%)")

✅ Ratios capped at sensible maximums

repayment_to_ndi_ratio after capping:
count    369.000000
mean       1.170062
std        1.252536
min        0.056400
25%        0.322300
50%        0.680000
75%        1.482700
max        5.000000
Name: repayment_to_ndi_ratio, dtype: float64

Distribution after capping:
  Above 0.30:  289 riders  (78%)
  Above 0.50:  226 riders  (61%)
  Above 1.00:  137 riders  (37%)
  Above 2.00:  68 riders  (18%)


In [55]:
print("DO HIGHER RATIOS PREDICT DEFAULT?")
print()

ratio_cols = [
    "loan_to_income_ratio",
    "loan_to_ndi_ratio",
    "repayment_to_ndi_ratio"
]

for col in ratio_cols:
    corr = feature_matrix[col].corr(
        feature_matrix["actual_default"].astype(float)
    )
    direction = "higher ratio = more default risk" if corr > 0 else "inverse"
    print(f"  {col:<30} correlation={corr:+.4f}  ({direction})")

print()
print("If correlations are positive — the ratios are working correctly")
print("Higher ratio = more unaffordable = higher default risk")

DO HIGHER RATIOS PREDICT DEFAULT?

  loan_to_income_ratio           correlation=-0.0473  (inverse)
  loan_to_ndi_ratio              correlation=-0.0336  (inverse)
  repayment_to_ndi_ratio         correlation=-0.0619  (inverse)

If correlations are positive — the ratios are working correctly
Higher ratio = more unaffordable = higher default risk


In [56]:
import json

# Final META_COLS
META_COLS = [
    "actual_default",
    "requested_amount",
    "requested_term_days",
    "requested_monthly_repayment",
]

FEATURE_COLS = [c for c in feature_matrix.columns
                if c not in META_COLS]

# Save
feature_matrix.to_csv("features.csv", index=False)
with open("feature_cols.json", "w") as f:
    json.dump(FEATURE_COLS, f, indent=2)

print(f"✅ Feature matrix saved")
print()
print(f"  Shape:         {feature_matrix.shape[0]} rows x {feature_matrix.shape[1]} columns")
print(f"  Features:      {len(FEATURE_COLS)}")
print(f"  Default rate:  {feature_matrix.actual_default.mean():.1%}")
print()
print("=" * 45)
print("  FEATURE ENGINEERING COMPLETE")
print("  Next step: Model Training")
print("=" * 45)

✅ Feature matrix saved

  Shape:         369 rows x 60 columns
  Features:      56
  Default rate:  3.0%

  FEATURE ENGINEERING COMPLETE
  Next step: Model Training


In [57]:
# Find extreme outliers
print("RIDERS WITH EXTREME RATIOS")
print()

extreme = feature_matrix[
    feature_matrix["repayment_to_ndi_ratio"] > 5
][[
    "requested_amount",
    "requested_term_days",
    "requested_monthly_repayment",
    "estimated_ndi",
    "repayment_to_ndi_ratio"
]]

print(f"Riders with ratio above 5: {len(extreme)}")
display(extreme)
print()

# What is causing extreme NDI?
# Find riders with very low NDI
low_ndi = feature_matrix[feature_matrix["estimated_ndi"] < 2000][[
    "monthly_income_estimate",
    "fuel_spend_monthly",
    "estimated_ndi"
]]
print(f"Riders with NDI below KES 2,000: {len(low_ndi)}")
display(low_ndi.head(10))

RIDERS WITH EXTREME RATIOS

Riders with ratio above 5: 0


,requested_amount,requested_term_days,requested_monthly_repayment,estimated_ndi,repayment_to_ndi_ratio



Riders with NDI below KES 2,000: 16


,monthly_income_estimate,fuel_spend_monthly,estimated_ndi
17,17301.7900,6098.59,1826.05
20,16237.3684,8521.34,1000.00
113,17002.8200,8620.39,1000.00
125,17339.5600,6877.61,1296.41
141,17004.0600,8645.43,1850.58
154,19360.1600,7675.35,1000.00
175,16551.7700,7997.66,1756.67
190,17266.6200,6071.16,1834.63
225,17480.3000,6962.56,1675.31
231,19259.7900,6571.17,1429.29


In [58]:
# Cap repayment_to_ndi_ratio at 5.0
# Anything above 5 means the same thing — completely unaffordable
# The exact value above 5 does not add information

feature_matrix["repayment_to_ndi_ratio"] = (
    feature_matrix["repayment_to_ndi_ratio"].clip(upper=5.0)
)

# Same for loan_to_ndi_ratio
feature_matrix["loan_to_ndi_ratio"] = (
    feature_matrix["loan_to_ndi_ratio"].clip(upper=10.0)
)

# Same for loan_to_income_ratio
feature_matrix["loan_to_income_ratio"] = (
    feature_matrix["loan_to_income_ratio"].clip(upper=10.0)
)

print("✅ Ratios capped at sensible maximums")
print()
print("repayment_to_ndi_ratio after capping:")
print(feature_matrix["repayment_to_ndi_ratio"].describe())
print()

# Now check distribution again
print("Distribution after capping:")
for threshold in [0.30, 0.50, 1.00, 2.00]:
    count = (feature_matrix["repayment_to_ndi_ratio"] > threshold).sum()
    pct   = count / len(feature_matrix) * 100
    print(f"  Above {threshold:.2f}:  {count} riders  ({pct:.0f}%)")

✅ Ratios capped at sensible maximums

repayment_to_ndi_ratio after capping:
count    369.000000
mean       1.170062
std        1.252536
min        0.056400
25%        0.322300
50%        0.680000
75%        1.482700
max        5.000000
Name: repayment_to_ndi_ratio, dtype: float64

Distribution after capping:
  Above 0.30:  289 riders  (78%)
  Above 0.50:  226 riders  (61%)
  Above 1.00:  137 riders  (37%)
  Above 2.00:  68 riders  (18%)


In [59]:
print("DO HIGHER RATIOS PREDICT DEFAULT?")
print()

ratio_cols = [
    "loan_to_income_ratio",
    "loan_to_ndi_ratio",
    "repayment_to_ndi_ratio"
]

for col in ratio_cols:
    corr = feature_matrix[col].corr(
        feature_matrix["actual_default"].astype(float)
    )
    direction = "higher ratio = more default risk" if corr > 0 else "inverse"
    print(f"  {col:<30} correlation={corr:+.4f}  ({direction})")

print()
print("If correlations are positive — the ratios are working correctly")
print("Higher ratio = more unaffordable = higher default risk")

DO HIGHER RATIOS PREDICT DEFAULT?

  loan_to_income_ratio           correlation=-0.0473  (inverse)
  loan_to_ndi_ratio              correlation=-0.0336  (inverse)
  repayment_to_ndi_ratio         correlation=-0.0619  (inverse)

If correlations are positive — the ratios are working correctly
Higher ratio = more unaffordable = higher default risk


In [60]:
import json

# Final META_COLS
META_COLS = [
    "actual_default",
    "requested_amount",
    "requested_term_days",
    "requested_monthly_repayment",
]

FEATURE_COLS = [c for c in feature_matrix.columns
                if c not in META_COLS]

# Save
feature_matrix.to_csv("features.csv", index=False)
with open("feature_cols.json", "w") as f:
    json.dump(FEATURE_COLS, f, indent=2)

print(f"✅ Feature matrix saved")
print()
print(f"  Shape:         {feature_matrix.shape[0]} rows x {feature_matrix.shape[1]} columns")
print(f"  Features:      {len(FEATURE_COLS)}")
print(f"  Default rate:  {feature_matrix.actual_default.mean():.1%}")
print()
print("=" * 45)
print("  FEATURE ENGINEERING COMPLETE")
print("  Next step: Model Training")
print("=" * 45)

✅ Feature matrix saved

  Shape:         369 rows x 60 columns
  Features:      56
  Default rate:  3.0%

  FEATURE ENGINEERING COMPLETE
  Next step: Model Training


In [49]:
return feature_matrix.to_json(orient="records")

SyntaxError: 'return' outside function (2005595527.py, line 1)

In [50]:
feature_matrix.to_json(orient="records")

'[{"rider_id":"RDR-00001","avg_daily_income":912.5884090909,"total_income_90d":80307.78,"income_volatility_cv":0.1798721025,"income_trend_90d":0.0409037669,"rain_season_dip":0.2474528483,"income_gap_max_days":1.0,"monthly_income_estimate":26769.26,"fuel_spend_monthly":8728.2,"fuel_to_income_ratio":0.3261,"sacco_contrib_monthly":3091.7,"family_remittance_monthly":0.0,"digital_loan_monthly_out":1194.76,"estimated_ndi":11949.36,"debt_service_ratio":0.1601,"max_safe_repayment":3584.81,"avg_mpesa_balance":20193.22,"min_mpesa_balance":2337.65,"zero_balance_days":0,"sacco_tenure_months":18,"sacco_contribution_rate":0.8889,"sacco_on_time_rate":0.875,"sacco_late_rate":0.1111,"sacco_avg_days_late":5.0,"sacco_cumulative_savings":25466.66,"sacco_guarantor_count":3,"sacco_loan_stress_ratio":0.0,"total_loans_taken":0,"loans_repaid_clean":0,"on_time_repayment_rate":0.75,"avg_days_early_late":0.0,"max_loan_repaid":0.0,"active_digital_loans":0,"digital_loan_frequency":0,"ever_defaulted":0,"restructured

In [61]:
# Check how many riders are in each feature family
print("RIDERS IN EACH FEATURE FAMILY")
print()
print(f"  income_features:    {income_features['rider_id'].nunique()}")
print(f"  expense_features:   {expense_features['rider_id'].nunique()}")
print(f"  sacco_features:     {sacco_features['rider_id'].nunique()}")
print(f"  loan_features:      {loan_features['rider_id'].nunique()}")
print(f"  behaviour_features: {behaviour_features['rider_id'].nunique()}")
print(f"  apps_clean:         {apps['rider_id'].nunique()}")

RIDERS IN EACH FEATURE FAMILY

  income_features:    1000
  expense_features:   1000
  sacco_features:     369
  loan_features:      1000
  behaviour_features: 1000
  apps_clean:         1000
